## Numerical Analysis - Spring semester 2026
# Serie 07 - Methods to solve non-linear system

First, we will need to import some of the usual packages. You will have to run this cell every time you restart your notebook.

In [ ]:
import numpy
import math
import time
import matplotlib.pyplot

<hr style="clear:both">

### Fixed-point iteration in Python

<div class="alert alert-success">
    
**Exercise 1:** Write two functions,
- `FixedPointIterationLast(g, x0, tol, i_max)`: it returns the last one-dimensional array `x` obtained by iterating `g` starting from `x0`;
- `FixedPointIterationAll(g, x0, tol, i_max)`: it returns a two-dimensional array `x` whose columns are the successive one-dimensional arrays obtained by iterating `g` starting from `x0`.
</div>


In [ ]:
def FixedPointIterationLast(g, x0, tol, i_max):
   x = x0
   delta = tol+1
   i = 0
   while i < i_max and delta > tol:
      x_new = g(x)
      delta = numpy.linalg.norm(x_new-x)
      x = x_new
      i = i+1
   if i == i_max:
      print("The maximal number of iterations has been reached, hence the method may not have converged: the distance between the last two iterates is ", delta)
   
   return x

def FixedPointIterationAll(g, x0, tol, i_max):
   i_max = i_max+1
   x = numpy.zeros((len(x0), i_max))
   x[:, 0] = x0
   x[:, 1] = g(x0)
   delta = numpy.linalg.norm(x[:, 1]-x[:, 0], 1)
   i = 2
   while i < i_max and delta > tol:
      x[:, i] = g(x[:, i-1])
      delta = numpy.linalg.norm(x[:, i]-x[:, i-1], 1)
      i = i+1
   if i == i_max:
      print("The maximal number of iterations has been reached, hence the method may not have converged: the distance between the last two iterates is ", delta)
   x = x[:, 0:i]
   
   return x

<hr style="clear:both">

### Newton’s iteration versus basic fixed-point iteration

<div class="alert alert-success">
    
**Exercise 2:** Using only numpy and math, implement the Python functions `g(x)` and `h(x)` that take as input a one-dimensional array of length 2 and return the images of that array under the functions `g` and `h`, defined respectively at the first and seventh points.
</div>

In [ ]:
def g(x):
   return [math.sin(x[0]+x[1])/3, math.cos(x[0]-x[1])/3]

def h(x):
   return (x-numpy.array([[math.sin(x[0]-x[1])-3, -math.cos(x[0]+x[1])],
                          [math.sin(x[0]-x[1]), math.cos(x[0]+x[1])-3]])
                          .dot([math.sin(x[0]+x[1])-3*x[0],
                                math.cos(x[0]-x[1])-3*x[1]])/
                                (9-3*(math.cos(x[0]+x[1])+math.sin(x[0]-x[1]))+
                                 2*math.cos(x[0]+x[1])*math.sin(x[0]-x[1])))

x0 = [0, 0]
tol = 1e-9
i_max = 50

#Compute run time
time_g = time.time()
x_g = FixedPointIterationAll(g, x0, tol, i_max)
time_g = time.time()-time_g
time_h = time.time()
x_h = FixedPointIterationAll(h, x0, tol, i_max)
time_h = time.time()-time_h

#Plot iterates
matplotlib.pyplot.figure()
matplotlib.pyplot.plot(x_g[0, :], x_g[1, :], 'go', label='g')
matplotlib.pyplot.plot(x_h[0, :], x_h[1, :], 'bo', label='h')
matplotlib.pyplot.xlabel('$x$')
matplotlib.pyplot.ylabel('$y$')
matplotlib.pyplot.legend()
ax = matplotlib.pyplot.gca()
ax.set_aspect('equal')

#Plot convergence speed
matplotlib.pyplot.figure()
matplotlib.pyplot.plot(numpy.log10(numpy.linalg.norm(x_g[:, 0:-2]-numpy.reshape(x_g[:, -1], (len(x_g[:, 0]), 1)), 1, 0)), 'go', label='g')
matplotlib.pyplot.plot(numpy.log10(numpy.linalg.norm(x_h[:, 0:-2]-numpy.reshape(x_h[:, -1], (len(x_h[:, 0]), 1)), 1, 0)), 'bo', label='h')
matplotlib.pyplot.xlabel('$i$')
matplotlib.pyplot.ylabel('$\log_{10}(||(x_i, y_i)-(x_*, y_*)||_1)$')
matplotlib.pyplot.legend()

<hr style="clear:both">

### Basins of attraction of Newton’s iteration

<div class="alert alert-success">
    
**Exercise 3:** Using only numpy, implement the Python function `g(x)` that takes as input a one-dimensional array of length 2 and returns the image of that array under the function `g` defined at the preceding point.
</div>

In [ ]:
def g(x):
   return (x-numpy.array([[x[0]**2-x[1]**2, 2*x[0]*x[1]],
                          [-2*x[0]*x[1], x[0]**2-x[1]**2]])
                          .dot([x[0]*(x[0]**2-3*x[1]**2)-1,
                                x[1]*(3*x[0]**2-x[1]**2)])/
                                (3*(x[0]**2+x[1]**2)**2))

#Tolerance on the distance between two successive iterates
tol = 1e-5
#Maximal number of iterations
i_max = 100
#Zeros of f
x1 = [1, 0]
x2 = [-0.5, 0.5*math.sqrt(3)]
x3 = [-0.5, -0.5*math.sqrt(3)]
#Grid
n = 1000
x = numpy.linspace(-1, 1, n)
#Basins of attraction
convergence = numpy.zeros((n, n))
for i in range(n):
   for j in range(n):
      z = FixedPointIterationLast(g, [x[i], x[j]], tol, i_max)
      if numpy.linalg.norm(z-x1) < tol:
         convergence[i, j] = 1
      elif numpy.linalg.norm(z-x2) < tol:
         convergence[i, j] = 2
      elif numpy.linalg.norm(z-x3) < tol:
         convergence[i, j] = 3

#Plot
matplotlib.pyplot.contourf(x, x, numpy.transpose(convergence), [0, 1, 2, 3])
matplotlib.pyplot.scatter([x1[0], x2[0], x3[0]], [x1[1], x2[1], x3[1]])
matplotlib.pyplot.xlim(-1, 1)
matplotlib.pyplot.ylim(-1, 1)
ax = matplotlib.pyplot.gca()
ax.set_aspect('equal')

<hr style="clear:both">

## The end

Congratulations! You have made it to the end of the third exercise notebook. 